# Simulación · Unidad III en profundidad
## Pruebas de bondad de ajuste, generación de variables aleatorias y modelamiento

Este cuaderno acompaña la presentación *Unidad3_Profundizacion_Bondad_Generacion_Modelamiento.pptx*. Cada ejemplo se resuelve **paso a paso**: primero a mano (con tablas intermedias), después con la función equivalente de SciPy para comprobar el resultado.

| Parte | Contenido |
|---|---|
| 0 | Preparación y funciones de la unidad |
| 1 | Pruebas de bondad: chi-cuadrada, Kolmogorov-Smirnov (una y dos muestras, Lilliefors) y Anderson-Darling; potencia |
| 2 | Generación: transformada inversa, convolución, aceptación y rechazo, composición |
| 3 | Modelamiento: ajuste al estilo Stat::Fit, sintaxis para ProModel/FlexSim, colas M/M/c, inventario (Q, R), línea de producción, pedido Monte Carlo |
| 4 | Casos: agrotrónica, domótica, inmótica, lectura de placas con YOLO (simulada y opcional real), encuesta de energía renovable |
| 5 | Los 18 ejercicios de la presentación, resueltos |

> Ejecuta en orden con *Entorno de ejecución → Ejecutar todas*. Todas las simulaciones usan semillas fijas, así que obtendrás los mismos números que en las diapositivas.

---
## 0. Preparación

In [ ]:
import math, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

MORADO, VERDE, AMBAR, CORAL, AZUL, GRIS, TINTA = "#6C4AB6", "#00A896", "#F2A900", "#E4572E", "#3A86C8", "#6B7280", "#1A1433"
PAL = [MORADO, VERDE, CORAL, AZUL, AMBAR]
sns.set_theme(style="whitegrid", palette=PAL)
plt.rcParams.update({"figure.figsize": (10, 4.2), "axes.titleweight": "bold", "axes.titlesize": 12})
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)

def decision(estadistico, critico, nombre="estadístico"):
    # Imprime la regla de decisión en lenguaje claro.
    if estadistico <= critico:
        print(f"{nombre} = {estadistico:.4f} ≤ crítico {critico:.4f} → NO se rechaza H0")
    else:
        print(f"{nombre} = {estadistico:.4f} > crítico {critico:.4f} → SE RECHAZA H0")

### Funciones de la unidad
Implementaciones propias (sin cajas negras) de las pruebas, los métodos de generación y los modelos. Lee el código: cada línea corresponde a una fórmula de la presentación.

In [ ]:
# ---------- pruebas de bondad ----------
def chi2_discreta(datos, pmf, valores, p_estimados=0, alfa=0.05, emin=5):
    """Chi-cuadrada para datos discretos. La última clase de `valores` acumula la cola (x ≥ último).
    Las clases de los extremos con E < emin se agrupan con su vecina."""
    datos = np.asarray(datos); n = len(datos)
    cl = []
    for k, x in enumerate(valores):
        cola = k == len(valores) - 1
        O = int(np.sum(datos >= x)) if cola else int(np.sum(datos == x))
        p = (1 - sum(c["p"] for c in cl)) if cola else pmf(x)
        cl.append(dict(lo=x, hi=None if cola else x, O=O, p=p))
    while len(cl) > 2 and n * cl[0]["p"] < emin:
        a = cl.pop(0); cl[0] = dict(lo=a["lo"], hi=cl[0]["hi"], O=a["O"] + cl[0]["O"], p=a["p"] + cl[0]["p"])
    while len(cl) > 2 and n * cl[-1]["p"] < emin:
        a = cl.pop(); cl[-1] = dict(lo=cl[-1]["lo"], hi=a["hi"], O=a["O"] + cl[-1]["O"], p=a["p"] + cl[-1]["p"])
    for c in cl:
        c["x"] = f"≥ {c['lo']}" if c["hi"] is None else (str(c["lo"]) if c["lo"] == c["hi"] else f"{c['lo']}–{c['hi']}")
        c["E"] = n * c["p"]; c["contrib"] = (c["O"] - c["E"]) ** 2 / c["E"]
    chi0 = sum(c["contrib"] for c in cl); gl = len(cl) - 1 - p_estimados
    crit = stats.chi2.ppf(1 - alfa, gl)
    return dict(n=n, clases=cl, chi0=chi0, gl=gl, crit=crit, p=1 - stats.chi2.cdf(chi0, gl), acepta=bool(chi0 <= crit))

def ks(datos, cdf, alfa=0.05):
    x = np.sort(np.asarray(datos)); n = len(x); F = cdf(x); i = np.arange(1, n + 1)
    dmas = i / n - F; dmenos = F - (i - 1) / n
    D = max(dmas.max(), dmenos.max()); crit = stats.kstwo.ppf(1 - alfa, n)
    return dict(n=n, x=x.tolist(), F=F.tolist(), dmas=dmas.tolist(), dmenos=dmenos.tolist(), Dmas=dmas.max(), Dmenos=dmenos.max(), D=D, crit=crit, acepta=bool(D <= crit))

AD_CRIT = {0.10: 1.933, 0.05: 2.492, 0.01: 3.857}  # distribución completamente especificada (caso 0)

def anderson_darling(datos, cdf, alfa=0.05):
    x = np.sort(np.asarray(datos)); n = len(x); F = np.clip(cdf(x), 1e-12, 1 - 1e-12); i = np.arange(1, n + 1)
    S = np.sum((2 * i - 1) * (np.log(F) + np.log(1 - F[::-1])))
    A2 = -n - S / n
    return dict(n=n, A2=A2, crit=AD_CRIT[alfa], acepta=bool(A2 <= AD_CRIT[alfa]), terminos=((2 * i - 1) * (np.log(F) + np.log(1 - F[::-1]))).tolist(), F=F.tolist(), x=x.tolist())

# ---------- generación ----------
def inv_exponencial(r, media): return -media * np.log(1 - np.asarray(r))
def inv_triangular(r, a, c, b):
    r = np.asarray(r, dtype=float); Fc = (c - a) / (b - a)
    return np.where(r < Fc, a + np.sqrt(r * (b - a) * (c - a)), b - np.sqrt((1 - r) * (b - a) * (b - c)))
def inv_discreta(r, valores, probs):
    acum = np.cumsum(probs); return [valores[int(np.searchsorted(acum, u, side="right"))] if u < acum[-1] else valores[-1] for u in np.atleast_1d(r)]
def erlang_convolucion(rs, k, media):
    """rs: arreglo (n, k). X = -(media/k) ln(prod r)."""
    rs = np.asarray(rs); return -(media / k) * np.log(np.prod(rs, axis=1))
def normal_suma12(rs, mu, sigma):
    return mu + sigma * (np.sum(rs, axis=1) - 6)
def aceptacion_rechazo(pares, f, a, b, M):
    out = []
    for r1, r2 in pares:
        x = a + (b - a) * r1; fx = f(x); acepta = r2 <= fx / M
        out.append(dict(r1=r1, r2=r2, x=x, fx=fx, umbral=fx / M, acepta=bool(acepta)))
    return out
def composicion(pares, pesos, medias):
    out = []; acum = np.cumsum(pesos)
    for r1, r2 in pares:
        j = int(np.searchsorted(acum, r1, side="right")); x = -medias[j] * np.log(1 - r2)
        out.append(dict(r1=r1, comp=j + 1, r2=r2, x=x))
    return out

# ---------- modelos ----------
def mm1_teoria(lam, mu):
    rho = lam / mu
    return dict(rho=rho, L=rho / (1 - rho), Lq=rho ** 2 / (1 - rho), W=1 / (mu - lam), Wq=lam / (mu * (mu - lam)), P0=1 - rho)

def simular_cola(n, lam, mu, rng, c=1):
    """Cola FIFO con c servidores, llegadas Poisson y servicio exponencial (tiempos en horas)."""
    ia = rng.exponential(1 / lam, n); sv = rng.exponential(1 / mu, n)
    llegada = np.cumsum(ia); libre = np.zeros(c); ini = np.empty(n); fin = np.empty(n)
    for i in range(n):
        j = int(np.argmin(libre)); ini[i] = max(llegada[i], libre[j]); fin[i] = ini[i] + sv[i]; libre[j] = fin[i]
    T = fin.max(); espera = ini - llegada
    return dict(llegada=llegada, ini=ini, fin=fin, servicio=sv, Wq=espera.mean(), W=(fin - llegada).mean(),
                util=sv.sum() / (c * T), Lq=espera.sum() / T, L=(fin - llegada).sum() / T)

def simular_inventario(dias, Q, R, inv0, dem_vals, dem_p, lt_vals, lt_p, K, h, s_costo, rng, detalle=False):
    inv = inv0; pedido = None; filas = []; costo = dict(pedido=0.0, mantener=0.0, faltante=0.0)
    for d in range(1, dias + 1):
        llega = 0
        if pedido and pedido[0] == d:
            inv += Q; llega = Q; pedido = None
        r = rng.random(); dem = inv_discreta([r], dem_vals, dem_p)[0]
        vend = min(inv, dem); falt = dem - vend; inv -= vend
        orden = ""
        if inv <= R and pedido is None:
            r2 = rng.random(); lt = inv_discreta([r2], lt_vals, lt_p)[0]
            pedido = (d + lt + 1, Q); costo["pedido"] += K; orden = f"Q={Q}, llega día {d + lt + 1}"
        costo["mantener"] += h * inv; costo["faltante"] += s_costo * falt
        if detalle:
            filas.append(dict(dia=d, recibe=llega, r=r, demanda=dem, vende=vend, faltante=falt, inv_final=inv, orden=orden))
    costo["total"] = sum(costo.values())
    return (filas, costo) if detalle else costo

def tiempo_pedido(rs, etapas):
    """rs: (n, k); etapas: lista de (a, c, b)."""
    return sum(inv_triangular(rs[:, j], *e) for j, e in enumerate(etapas))

---
# Parte 1 · Pruebas de bondad de ajuste

## 1.1 Lógica de la prueba de hipótesis

* $H_0$: los datos provienen de la distribución propuesta $F_0$.
* $H_1$: no provienen de $F_0$.
* **Error tipo I** ($\alpha$): rechazar $H_0$ cuando es cierta. Lo fijamos (5 %).
* **Error tipo II** ($\beta$): no rechazar $H_0$ cuando es falsa. **Potencia** $= 1-\beta$.
* **Valor p**: probabilidad, bajo $H_0$, de obtener un estadístico tan extremo o más que el observado. Si $p < \alpha$ se rechaza.

> «No se rechaza» **no** significa «se demuestra». Con muestras pequeñas casi todo pasa (ver 1.9).

In [ ]:
gl, chi_obs = 4, 7.2
x = np.linspace(0, 16, 400); y = stats.chi2.pdf(x, gl)
crit = stats.chi2.ppf(0.95, gl); p = stats.chi2.sf(chi_obs, gl)
fig, ax = plt.subplots()
ax.plot(x, y, color=MORADO, lw=2, label=f"χ² con {gl} gl")
ax.fill_between(x, y, where=x >= crit, color=CORAL, alpha=.35, label=f"región de rechazo (α = 5 %, crítico {crit:.2f})")
ax.fill_between(x, y, where=x >= chi_obs, color=AMBAR, alpha=.5, label=f"valor p = P(χ² ≥ {chi_obs}) = {p:.3f}")
ax.axvline(chi_obs, color=TINTA, ls="--"); ax.set_title("Valor p y región de rechazo"); ax.legend(); plt.show()

## 1.2 Chi-cuadrada para datos discretos: vehículos por minuto en la portería (YOLO)

La cámara con YOLO cuenta vehículos por minuto durante 100 minutos. ¿Siguen una **Poisson**?

Pasos:
1. Estimar $\hat\lambda = \bar x$ (1 parámetro estimado).
2. Probabilidades $p_i = e^{-\lambda}\lambda^x/x!$; la última clase acumula la cola $P(X \ge 8)$.
3. Frecuencias esperadas $E_i = n p_i$. **Agrupar** clases con $E_i < 5$.
4. $\chi^2_0 = \sum (O_i - E_i)^2 / E_i$.
5. $gl = k - 1 - m$ (k clases tras agrupar, m parámetros estimados).

In [ ]:
rng = np.random.default_rng(2026)
veh = rng.poisson(3, 100)
lam = veh.mean()
print("Primeros 20 datos:", veh[:20])
print(f"n = {len(veh)},  λ̂ = media = {lam:.2f},  varianza = {veh.var(ddof=1):.2f}  (en la Poisson media ≈ varianza)")

# Paso 2-3 SIN agrupar, para ver por qué hay que agrupar
tabla = pd.DataFrame({"x": list(range(8)) + ["≥8"]})
tabla["O"] = [int((veh == k).sum()) for k in range(8)] + [int((veh >= 8).sum())]
tabla["p"] = [stats.poisson.pmf(k, lam) for k in range(8)] + [stats.poisson.sf(7, lam)]
tabla["E"] = len(veh) * tabla["p"]
tabla

In [ ]:
# Paso 3-5 con la función propia (agrupa automáticamente)
res = chi2_discreta(veh, lambda x: stats.poisson.pmf(x, lam), list(range(0, 9)), p_estimados=1)
t = pd.DataFrame(res["clases"])[["x", "O", "p", "E", "contrib"]]
t.loc["Σ"] = ["", t.O.sum(), t.p.sum(), t.E.sum(), t.contrib.sum()]
display(t)
print(f"gl = {len(res['clases'])} clases − 1 − 1 parámetro = {res['gl']}")
decision(res["chi0"], res["crit"], "χ²₀"); print(f"valor p = {res['p']:.4f}")

In [ ]:
# Comprobación con SciPy: ddof = parámetros estimados
O = np.array([c["O"] for c in res["clases"]]); E = np.array([c["E"] for c in res["clases"]])
print(stats.chisquare(O, E, ddof=1))
fig, ax = plt.subplots()
xs = np.arange(len(O)); ax.bar(xs - .2, O, .4, label="Observado", color=MORADO); ax.bar(xs + .2, E, .4, label="Esperado Poisson", color=VERDE)
ax.set_xticks(xs, [c["x"] for c in res["clases"]]); ax.set_xlabel("vehículos por minuto"); ax.legend(); ax.set_title("Observado vs. esperado"); plt.show()

## 1.3 Chi-cuadrada con datos continuos y clases equiprobables

Con datos continuos hay que construir clases. La opción recomendada son **k clases equiprobables**: los límites son los cuantiles $b_j = F^{-1}(j/k)$, así cada clase tiene $E = n/k$.

Datos: 200 tiempos de servicio (min). Comparamos una **exponencial** (1 parámetro) contra una **gamma** (2 parámetros).

In [ ]:
d200 = np.round(np.random.default_rng(7).gamma(2.5, 2.0, 200), 2)
print(pd.Series(d200).describe().round(3).to_string())

def chi_equiprobable(x, dist, par, k=10, npar=0):
    bordes = dist.ppf(np.linspace(0, 1, k + 1), *par)   # cuantiles j/k
    O = np.histogram(x, bins=bordes)[0]; E = len(x) / k
    contrib = (O - E) ** 2 / E; gl = k - 1 - npar
    tabla = pd.DataFrame({"desde": bordes[:-1], "hasta": bordes[1:], "O": O, "E": E, "(O−E)²/E": contrib})
    return tabla, contrib.sum(), gl

pe = stats.expon.fit(d200, floc=0); pg = stats.gamma.fit(d200, floc=0)
print(f"\nExponencial: media = {pe[1]:.3f}")
print(f"Gamma: forma α = {pg[0]:.3f}, escala β = {pg[2]:.3f}")
for nombre, dist, par, m in [("Exponencial", stats.expon, pe, 1), ("Gamma", stats.gamma, pg, 2)]:
    tab, chi, gl = chi_equiprobable(d200, dist, par, 10, m)
    print(f"\n=== {nombre} ===")
    display(tab.round(3))
    decision(chi, stats.chi2.ppf(.95, gl), f"χ²₀ ({gl} gl)"); print(f"valor p = {stats.chi2.sf(chi, gl):.4g}")

In [ ]:
fig, ax = plt.subplots()
ax.hist(d200, bins=20, density=True, color=MORADO, alpha=.45, edgecolor="white", label="datos")
t = np.linspace(0, d200.max(), 300)
ax.plot(t, stats.expon.pdf(t, *pe), color=CORAL, lw=2, label="exponencial")
ax.plot(t, stats.gamma.pdf(t, *pg), color=VERDE, lw=2, label="gamma")
ax.set_xlabel("minutos"); ax.legend(); ax.set_title("La gamma sigue la forma de la joroba; la exponencial no"); plt.show()

## 1.4 Chi-cuadrada de independencia: encuesta de energía renovable

¿La disposición a instalar paneles solares depende del estrato?

$$E_{ij} = \frac{(\text{total fila } i)(\text{total columna } j)}{n}, \qquad gl = (f-1)(c-1), \qquad V = \sqrt{\frac{\chi^2}{n\,(\min(f,c)-1)}}$$

In [ ]:
tab = pd.DataFrame([[38, 30, 32], [52, 34, 24], [46, 26, 18]], index=["Estrato 1-2", "Estrato 3-4", "Estrato 5-6"], columns=["Sí", "Tal vez", "No"])
n = tab.values.sum()
E = np.outer(tab.sum(1), tab.sum(0)) / n
E = pd.DataFrame(E, index=tab.index, columns=tab.columns)
contrib = (tab - E) ** 2 / E
print("Observado:"); display(tab)
print("Esperado:"); display(E.round(2))
print("Contribuciones (O−E)²/E:"); display(contrib.round(3))
chi = contrib.values.sum(); gl = (tab.shape[0] - 1) * (tab.shape[1] - 1)
decision(chi, stats.chi2.ppf(.95, gl), f"χ²₀ ({gl} gl)")
V = math.sqrt(chi / (n * (min(tab.shape) - 1)))
print(f"valor p = {stats.chi2.sf(chi, gl):.4f};  V de Cramér = {V:.3f} (asociación débil si < 0.2)")
print("SciPy:", stats.chi2_contingency(tab, correction=False)[:3])

## 1.5 Kolmogorov-Smirnov (una muestra): 20 tiempos entre llegadas

$$D^+ = \max_i\left(\tfrac{i}{n} - F_0(x_{(i)})\right), \quad D^- = \max_i\left(F_0(x_{(i)}) - \tfrac{i-1}{n}\right), \quad D = \max(D^+, D^-)$$

$H_0$: exponencial con media 4 (completamente especificada). Crítico exacto para $n=20$, $\alpha=0.05$: 0.294 (aprox. $1.36/\sqrt n$ para n grande).

In [ ]:
s20 = np.round(np.random.default_rng(31).exponential(4, 20), 2)
F0 = lambda x: 1 - np.exp(-np.asarray(x) / 4)
r = ks(s20, F0)
t = pd.DataFrame({"i": np.arange(1, 21), "x(i)": r["x"], "F0(x)": r["F"], "i/n": np.arange(1, 21) / 20, "D+": r["dmas"], "D−": r["dmenos"]})
display(t.round(4))
print(f"D+ = {r['Dmas']:.4f}   D− = {r['Dmenos']:.4f}   D = {r['D']:.4f}")
decision(r["D"], r["crit"], "D")
print("SciPy:", stats.kstest(s20, "expon", args=(0, 4)))

In [ ]:
xs = np.sort(s20); n = len(xs)
fig, ax = plt.subplots()
ax.step(np.r_[0, xs], np.r_[0, np.arange(1, n + 1) / n], where="post", color=MORADO, lw=2, label="F empírica")
t = np.linspace(0, xs.max() * 1.05, 300); ax.plot(t, F0(t), color=VERDE, lw=2, label="F0 exponencial(4)")
i = np.argmax(np.maximum(r["dmas"], r["dmenos"]))
ax.vlines(xs[i], F0(xs[i]), (i + 1) / n if r["dmas"][i] >= r["dmenos"][i] else i / n, color=CORAL, lw=3, label=f"D = {r['D']:.3f}")
ax.legend(); ax.set_title("K-S: la mayor distancia vertical entre las acumuladas"); plt.show()

## 1.6 Kolmogorov-Smirnov para dos muestras: ¿dos sensores de humedad miden igual?

$D_{n,m} = \max_x |F_A(x) - F_B(x)|$. Crítico aproximado al 5 %: $1.36\sqrt{(n+m)/(nm)}$.

In [ ]:
rng = np.random.default_rng(44)
A = np.round(rng.normal(32, 4, 25), 1); B = np.round(rng.normal(35, 4, 30), 1)
n, m = len(A), len(B)
malla = np.sort(np.r_[A, B])
FA = np.searchsorted(np.sort(A), malla, side="right") / n
FB = np.searchsorted(np.sort(B), malla, side="right") / m
dif = np.abs(FA - FB); Dnm = dif.max(); crit = 1.36 * math.sqrt((n + m) / (n * m))
print(f"media A = {A.mean():.2f}, media B = {B.mean():.2f}")
print(f"D = {Dnm:.4f} en x = {malla[dif.argmax()]}"); decision(Dnm, crit, "D")
print("SciPy:", stats.ks_2samp(A, B))
fig, ax = plt.subplots()
ax.step(malla, FA, where="post", label="sensor A", color=MORADO); ax.step(malla, FB, where="post", label="sensor B", color=VERDE)
ax.axvline(malla[dif.argmax()], color=CORAL, ls="--"); ax.legend(); ax.set_xlabel("humedad del suelo (%)"); ax.set_title("K-S dos muestras"); plt.show()

## 1.7 Lilliefors: K-S cuando los parámetros se estiman

Si $\mu$ y $\sigma$ se estiman con la misma muestra, la tabla K-S clásica es **demasiado permisiva**. Lilliefors obtuvo los críticos por simulación. Aquí los reproducimos con Monte Carlo: así se ve **por qué** cambian.

In [ ]:
def D_estimado(x):
    mu, sd = x.mean(), x.std(ddof=1)
    return ks(x, lambda v: stats.norm.cdf(v, mu, sd))["D"]

n, reps = 30, 3000
rng = np.random.default_rng(123)
Ds = np.array([D_estimado(rng.normal(0, 1, n)) for _ in range(reps)])
crit_lillie = np.quantile(Ds, 0.95)
print(f"n = {n}: crítico K-S clásico = {stats.kstwo.ppf(.95, n):.4f}   crítico Lilliefors (Monte Carlo) = {crit_lillie:.4f}")
print("El crítico correcto es mucho menor: con la tabla clásica casi nunca se rechazaría la normal.")
try:
    from statsmodels.stats.diagnostic import lilliefors
    T30 = np.round(np.random.default_rng(9).normal(24, 2, 30), 1)
    print("statsmodels lilliefors (temperaturas):", lilliefors(T30, dist="norm"))
except ImportError:
    print("(statsmodels no está instalado; en Colab sí viene)")

## 1.8 Anderson-Darling

$$A^2 = -n - \frac{1}{n}\sum_{i=1}^{n}(2i-1)\left[\ln F_0(x_{(i)}) + \ln\big(1 - F_0(x_{(n+1-i)})\big)\right]$$

Da más peso a las **colas** que K-S. Críticos al 5 %:

| Caso | Estadístico | Crítico |
|---|---|---|
| Distribución completamente especificada | $A^2$ | 2.492 |
| Normal con $\mu,\sigma$ estimados | $A^* = A^2(1 + 0.75/n + 2.25/n^2)$ | 0.752 |
| Exponencial con media estimada | $A^2(1 + 0.6/n)$ | 1.321 |

In [ ]:
# a) Caso especificado: los mismos 20 tiempos frente a Exp(4)
r = anderson_darling(s20, F0)
t = pd.DataFrame({"i": np.arange(1, 21), "x(i)": r["x"], "F0(x(i))": r["F"], "F0(x(n+1−i))": r["F"][::-1], "(2i−1)[ln F + ln(1−F)]": r["terminos"]})
display(t.round(4))
print(f"Σ = {sum(r['terminos']):.4f}  →  A² = −20 − Σ/20 = {r['A2']:.4f}")
decision(r["A2"], 2.492, "A²")

In [ ]:
# b) Normal con parámetros estimados: temperatura del invernadero (30 lecturas)
T30 = np.round(np.random.default_rng(9).normal(24, 2, 30), 1)
mu, sd = T30.mean(), T30.std(ddof=1)
r = anderson_darling(T30, lambda x: stats.norm.cdf(x, mu, sd))
A_est = r["A2"] * (1 + 0.75 / 30 + 2.25 / 900)
print(f"μ̂ = {mu:.3f}  σ̂ = {sd:.3f}  A² = {r['A2']:.4f}  A* = {A_est:.4f}")
decision(A_est, 0.752, "A*")
sa = stats.anderson(T30, "norm")
print(f"SciPy: A² = {sa.statistic:.4f}; críticos {dict(zip(sa.significance_level, sa.critical_values))}")
print("Nota: SciPy compara A² sin corregir contra críticos ya ajustados por n; la decisión es la misma.")

## 1.9 Potencia de las pruebas (Monte Carlo)

Repetimos el experimento muchas veces:
* Bajo $H_0$ (datos realmente exponenciales) la tasa de rechazo debe ser ≈ 5 % (error tipo I).
* Bajo $H_1$ (datos gamma) la tasa de rechazo es la **potencia**.

In [ ]:
def tasa_rechazo(generador, n, reps=600, semilla=0):
    rng = np.random.default_rng(semilla); k = max(4, round(math.sqrt(n)))
    cdf = lambda x: 1 - np.exp(-x / 4.5)
    bordes = np.r_[-4.5 * np.log(1 - np.arange(k) / k), np.inf]
    cc, ck = stats.chi2.ppf(.95, k - 1), stats.kstwo.ppf(.95, n); rc = rk = ra = 0
    for _ in range(reps):
        x = generador(rng, n); O = np.histogram(x, bins=bordes)[0]
        rc += ((O - n / k) ** 2 / (n / k)).sum() > cc
        rk += ks(x, cdf)["D"] > ck
        ra += anderson_darling(x, cdf)["A2"] > 2.492
    return rc / reps, rk / reps, ra / reps

ns = [10, 20, 30, 50, 100]
t0 = time.time()
pot = pd.DataFrame([tasa_rechazo(lambda r, n: r.gamma(2.5, 1.8, n), n, semilla=10 + n) for n in ns], index=ns, columns=["χ²", "K-S", "A-D"])
err1 = pd.DataFrame([tasa_rechazo(lambda r, n: r.exponential(4.5, n), n, semilla=500 + n) for n in ns], index=ns, columns=["χ²", "K-S", "A-D"])
print(f"({time.time() - t0:.1f} s)"); print("Potencia (H1 gamma):"); display(pot); print("Error tipo I (H0 cierta):"); display(err1)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
pot.plot(marker="o", ax=ax[0], title="Potencia frente a n"); ax[0].set_xlabel("n")
err1.plot(marker="o", ax=ax[1], title="Error tipo I (debe rondar 0.05)"); ax[1].axhline(.05, color=GRIS, ls="--"); ax[1].set_ylim(0, .15)
plt.show()

**Lectura:** H₀ (exponencial) y H₁ (gamma) tienen la misma media 4.5, así que la diferencia está en la forma. Con n = 10–20 ninguna prueba detecta el error más de un tercio de las veces; en este experimento χ² rechaza algo más con n pequeño, y desde n = 50 A-D es la más potente porque pondera las colas. El error tipo I de las tres ronda el 5 %, como debe ser. Conclusión práctica: **al menos 30–50 datos** antes de confiar en un «no se rechaza».

## 1.10 ¿Cuál prueba usar?

| | χ² | K-S | A-D |
|---|---|---|---|
| Datos | discretos o continuos agrupados | continuos | continuos |
| n mínimo | grande (E ≥ 5) | cualquiera | cualquiera |
| Sensibilidad | global | centro | colas |
| Parámetros estimados | resta gl | Lilliefors | corrección por n |

---
# Parte 2 · Generación de variables aleatorias

Todos los métodos parten de números $r \sim U(0,1)$. Usamos los mismos seis números de la presentación:
$R = 0.2311,\ 0.6068,\ 0.4860,\ 0.8913,\ 0.7621,\ 0.4565$.

Después de cada método **validamos**: generamos 20 000 valores y comparamos media, varianza y prueba K-S contra la teórica.

In [ ]:
R = np.array([0.2311, 0.6068, 0.4860, 0.8913, 0.7621, 0.4565])
def validar(muestra, dist, nombre):
    # Compara momentos y aplica K-S (continuas) a una muestra grande generada.
    fila = {"método": nombre, "media muestral": muestra.mean(), "media teórica": dist.mean(),
            "var muestral": muestra.var(ddof=1), "var teórica": dist.var()}
    if isinstance(dist.dist, stats.rv_continuous):
        fila["p K-S"] = stats.kstest(muestra, dist.cdf).pvalue
    return fila
validaciones = []

## 2.1 Transformada inversa

Si $F$ es continua y estrictamente creciente, $X = F^{-1}(R)$ tiene distribución $F$, porque
$P(X \le x) = P(F^{-1}(R) \le x) = P(R \le F(x)) = F(x)$.

### a) Exponencial
$r = 1 - e^{-x/\beta} \Rightarrow x = -\beta\ln(1-r)$

In [ ]:
beta = 4
t = pd.DataFrame({"r": R, "1−r": 1 - R, "ln(1−r)": np.log(1 - R), "x = −4 ln(1−r)": inv_exponencial(R, beta)})
display(t.round(4))
big = inv_exponencial(np.random.default_rng(1).random(20000), beta)
validaciones.append(validar(big, stats.expon(scale=beta), "Inversa · exponencial"))

### b) Weibull: vida útil de una bomba de riego ($k = 1.8$, $\lambda = 900$ h)
$F(x) = 1 - e^{-(x/\lambda)^k} \Rightarrow x = \lambda\,[-\ln(1-r)]^{1/k}$

In [ ]:
k, lam_w = 1.8, 900
t = pd.DataFrame({"r": R, "−ln(1−r)": -np.log(1 - R), "^(1/k)": (-np.log(1 - R)) ** (1 / k), "x (h)": lam_w * (-np.log(1 - R)) ** (1 / k)})
display(t.round(4))
print(f"P(falla antes de 1000 h) = 1 − exp(−(1000/900)^1.8) = {1 - math.exp(-(1000 / 900) ** 1.8):.4f}")
print(f"MTTF = λ Γ(1 + 1/k) = {lam_w * math.gamma(1 + 1 / k):.1f} h")
big = lam_w * (-np.log(1 - np.random.default_rng(2).random(20000))) ** (1 / k)
validaciones.append(validar(big, stats.weibull_min(k, scale=lam_w), "Inversa · Weibull"))

### c) Pareto (tamaño de archivos de video, cola pesada): $x = x_m (1-r)^{-1/a}$

In [ ]:
xm, a = 2.0, 3.0
print(pd.DataFrame({"r": R, "x (MB)": xm * (1 - R) ** (-1 / a)}).round(4).to_string())
big = xm * (1 - np.random.default_rng(3).random(20000)) ** (-1 / a)
validaciones.append(validar(big, stats.pareto(a, scale=xm), "Inversa · Pareto"))

### d) Discretas: geométrica (fórmula cerrada) y Poisson (búsqueda en la acumulada)

* Geométrica (número de intentos hasta el primer éxito, $p = 0.6$): $x = \lceil \ln(1-r)/\ln(1-p) \rceil$.
* Poisson ($\lambda = 3$): se busca el menor $x$ con $F(x) \ge r$.

In [ ]:
p = 0.6
geo = pd.DataFrame({"r": R, "ln(1−r)/ln(0.4)": np.log(1 - R) / np.log(1 - p)})
geo["x = techo"] = np.ceil(geo.iloc[:, 1]).astype(int)
display(geo.round(4))
acum = pd.DataFrame({"x": range(9), "p(x)": stats.poisson.pmf(range(9), 3), "F(x)": stats.poisson.cdf(range(9), 3)})
display(acum.round(4))
def poisson_busqueda(r, lam):
    x, p_, F = 0, math.exp(-lam), math.exp(-lam)
    while r > F:              # se avanza hasta que F(x) ≥ r
        x += 1; p_ *= lam / x; F += p_
    return x
print("Poisson generada:", [poisson_busqueda(r, 3) for r in R])
big_g = np.ceil(np.log(1 - np.random.default_rng(4).random(20000)) / np.log(1 - p))
big_p = np.array([poisson_busqueda(r, 3) for r in np.random.default_rng(5).random(20000)])
validaciones += [validar(big_g, stats.geom(p), "Inversa · geométrica"), validar(big_p, stats.poisson(3), "Inversa · Poisson")]

### e) Distribución empírica con interpolación lineal

Cuando ninguna distribución teórica ajusta, se usa la **empírica**: con los datos ordenados $x_{(1)},\dots,x_{(n)}$, posición $= r(n-1)$ y se interpola entre los dos puntos vecinos.

In [ ]:
emp = np.sort(np.array([2.1, 3.4, 3.9, 4.6, 5.0, 5.8, 6.3, 7.7, 8.9, 11.2]))
def empirica_inv(u, datos=emp):
    n = len(datos); pos = u * (n - 1); i = min(int(pos), n - 2)
    return datos[i] + (pos - i) * (datos[i + 1] - datos[i])
filas = []
for r in R:
    pos = r * 9; i = int(pos)
    filas.append(dict(r=r, posicion=pos, i=i, x_i=emp[i], x_i1=emp[i + 1], x=empirica_inv(r)))
display(pd.DataFrame(filas).round(4))

## 2.2 Convolución: sumar variables

### a) Erlang (k = 3, media 6): suma de 3 exponenciales de media 2
$X = -\frac{\text{media}}{k}\ln\left(\prod_{j=1}^{k} r_j\right)$

In [ ]:
grupos = np.array([[0.2311, 0.6068, 0.4860], [0.8913, 0.7621, 0.4565], [0.0185, 0.8214, 0.4447], [0.6154, 0.7919, 0.9218]])
t = pd.DataFrame(grupos, columns=["r1", "r2", "r3"]); t["Π r"] = grupos.prod(1); t["x = −2 ln Π"] = erlang_convolucion(grupos, 3, 6)
display(t.round(4))
big = erlang_convolucion(np.random.default_rng(6).random((20000, 3)), 3, 6)
validaciones.append(validar(big, stats.gamma(3, scale=2), "Convolución · Erlang"))

### b) Binomial como suma de Bernoulli y normal como suma de 12 uniformes
* Binomial(10, 0.1): cuenta cuántos de 10 números son menores que 0.1.
* Normal: $Z = \sum_{j=1}^{12} r_j - 6$ (media 0, varianza 12·1/12 = 1, por el teorema central del límite).

In [ ]:
br = np.random.default_rng(3).random((5, 10))
print("Binomial: ", (br < 0.1).sum(1))
big_b = (np.random.default_rng(7).random((20000, 10)) < 0.1).sum(1)
big_n = normal_suma12(np.random.default_rng(8).random((20000, 12)), 0, 1)
validaciones += [validar(big_b, stats.binom(10, .1), "Convolución · binomial"), validar(big_n, stats.norm(0, 1), "Convolución · normal (12 U)")]
print(f"Curtosis de la suma de 12: {stats.kurtosis(big_n):.3f} (normal = 0); la cola se trunca en ±6")

## 2.3 Aceptación y rechazo

Se usa cuando $F^{-1}$ no tiene fórmula. Con una envolvente $c\,g(x) \ge f(x)$:
1. Generar $Y \sim g$.
2. Generar $r \sim U(0,1)$.
3. Aceptar $Y$ si $r \le f(Y) / (c\,g(Y))$; si no, volver a 1.

Eficiencia $= 1/c$ (fracción aceptada).

### a) Beta(2, 2): $f(x) = 6x(1-x)$ en [0, 1], envolvente rectangular de altura M = 1.5

In [ ]:
pares = [(0.2311, 0.6068), (0.4860, 0.8913), (0.7621, 0.4565), (0.0185, 0.8214), (0.4447, 0.6154), (0.7919, 0.9218)]
display(pd.DataFrame(aceptacion_rechazo(pares, lambda x: 6 * x * (1 - x), 0, 1, 1.5)).round(4))
rng = np.random.default_rng(9); U = rng.random((30000, 2)); x = U[:, 0]; ok = U[:, 1] <= 6 * x * (1 - x) / 1.5
print(f"Aceptados: {ok.mean():.4f}  (teórico 1/M = {1 / 1.5:.4f})")
validaciones.append(validar(x[ok], stats.beta(2, 2), "Aceptación-rechazo · Beta(2,2)"))
fig, ax = plt.subplots()
ax.scatter(x[:3000][ok[:3000]], (U[:3000, 1] * 1.5)[ok[:3000]], s=4, color=VERDE, label="aceptados")
ax.scatter(x[:3000][~ok[:3000]], (U[:3000, 1] * 1.5)[~ok[:3000]], s=4, color=CORAL, label="rechazados")
t = np.linspace(0, 1, 200); ax.plot(t, 6 * t * (1 - t), color=TINTA, lw=2, label="f(x)"); ax.axhline(1.5, color=MORADO, ls="--", label="M = 1.5")
ax.legend(loc="upper right"); ax.set_title("Aceptación y rechazo para Beta(2, 2)"); plt.show()

### b) Normal a partir de la exponencial
Media normal $f(y) = \sqrt{2/\pi}\,e^{-y^2/2}$, envolvente $g(y) = e^{-y}$, $c = \sqrt{2e/\pi} = 1.3155$.
Se acepta si $r_2 \le e^{-(y-1)^2/2}$; luego se asigna un signo al azar.

In [ ]:
c = math.sqrt(2 * math.e / math.pi)
filas = []
for r1, r2 in pares:
    y = -math.log(1 - r1); umbral = math.exp(-(y - 1) ** 2 / 2)
    filas.append(dict(r1=r1, y=y, r2=r2, umbral=umbral, acepta=r2 <= umbral))
display(pd.DataFrame(filas).round(4))
rng = np.random.default_rng(10); y = rng.exponential(1, 40000); ok = rng.random(40000) <= np.exp(-(y - 1) ** 2 / 2)
z = y[ok] * np.where(rng.random(ok.sum()) < .5, -1, 1)
print(f"c = {c:.4f}; eficiencia teórica 1/c = {1 / c:.4f}; observada = {ok.mean():.4f}")
validaciones.append(validar(z, stats.norm(0, 1), "Aceptación-rechazo · normal"))

## 2.4 Composición: mezclas

$f(x) = \sum_j p_j f_j(x)$. Con $r_1$ se elige la componente, con $r_2$ se genera de ella.

Hiperexponencial: 70 % de llamadas cortas (media 2 min) y 30 % largas (media 8 min). Media $= 0.7·2 + 0.3·8 = 3.8$.

In [ ]:
display(pd.DataFrame(composicion(pares, [0.7, 0.3], [2, 8])).round(4))
rng = np.random.default_rng(11); U = rng.random((20000, 2)); medias = np.where(U[:, 0] < .7, 2, 8); big = -medias * np.log(1 - U[:, 1])
m = 0.7 * 2 + 0.3 * 8; var = 0.7 * 2 * 2 ** 2 + 0.3 * 2 * 8 ** 2 - m ** 2
print(f"media: {big.mean():.3f} (teórica {m}); varianza: {big.var():.3f} (teórica {var:.2f}); CV = {big.std() / big.mean():.3f} > 1")
fig, ax = plt.subplots(); ax.hist(big, bins=80, density=True, color=MORADO, alpha=.5, range=(0, 40))
t = np.linspace(0, 40, 400); ax.plot(t, .7 * stats.expon.pdf(t, scale=2) + .3 * stats.expon.pdf(t, scale=8), color=CORAL, lw=2)
ax.set_title("Hiperexponencial por composición"); plt.show()

## 2.5 Resumen de validación y velocidad

In [ ]:
display(pd.DataFrame(validaciones).set_index("método").round(4))
N = 200000; rng = np.random.default_rng(0); tiempos = {}
t0 = time.perf_counter(); -np.log(1 - rng.random(N)); tiempos["inversa exponencial"] = time.perf_counter() - t0
t0 = time.perf_counter(); erlang_convolucion(rng.random((N, 3)), 3, 6); tiempos["convolución Erlang"] = time.perf_counter() - t0
t0 = time.perf_counter(); u = rng.random((int(N * 1.6), 2)); u[:, 0][u[:, 1] <= 4 * u[:, 0] * (1 - u[:, 0])]; tiempos["aceptación-rechazo Beta"] = time.perf_counter() - t0
t0 = time.perf_counter(); u = rng.random((N, 2)); -np.where(u[:, 0] < .7, 2, 8) * np.log(1 - u[:, 1]); tiempos["composición"] = time.perf_counter() - t0
pd.Series(tiempos, name="segundos para 200 000 valores").round(4)

Todas las p de K-S son altas: los generadores producen la distribución correcta. La inversa es la más rápida cuando $F^{-1}$ existe en forma cerrada.

---
# Parte 3 · Modelamiento

## 3.1 Ajuste automático al estilo Stat::Fit

Stat::Fit (incluido con ProModel) hace: estadística descriptiva → estimación de parámetros por máxima verosimilitud → pruebas χ², K-S y A-D → ranking → exportación a la sintaxis del simulador. Reproducimos ese flujo con SciPy sobre los 200 tiempos de servicio.

In [ ]:
x = d200
desc = pd.Series({"n": len(x), "media": x.mean(), "mediana": np.median(x), "desv. estándar": x.std(ddof=1), "CV": x.std(ddof=1) / x.mean(),
                  "asimetría": stats.skew(x), "curtosis exceso": stats.kurtosis(x), "mín": x.min(), "máx": x.max()})
print(desc.round(4).to_string())
print("\nPista: CV ≈ 1 sugiere exponencial; CV < 1 con asimetría positiva sugiere gamma, Weibull, lognormal o Erlang.")

In [ ]:
candidatas = {
    "Exponencial": (stats.expon, dict(floc=0)),
    "Gamma": (stats.gamma, dict(floc=0)),
    "Weibull": (stats.weibull_min, dict(floc=0)),
    "Lognormal": (stats.lognorm, dict(floc=0)),
    "Normal": (stats.norm, {}),
    "Uniforme": (stats.uniform, {}),
}
filas = []
for nombre, (dist, fijo) in candidatas.items():
    par = dist.fit(x, **fijo); npar = len(par) - len(fijo)
    F = lambda v, d=dist, p=par: d.cdf(v, *p)
    Dks = ks(x, F)["D"]; A2 = anderson_darling(x, F)["A2"]
    bordes = dist.ppf(np.linspace(0, 1, 11), *par); O = np.histogram(x, bins=bordes)[0]; chi = ((O - 20) ** 2 / 20).sum()
    ll = dist.logpdf(x, *par).sum()
    filas.append(dict(distribución=nombre, parámetros=np.round(par, 3).tolist(), k=npar, chi2=chi, p_chi2=stats.chi2.sf(chi, 9 - npar),
                      D_KS=Dks, A2=A2, AIC=2 * npar - 2 * ll))
ranking = pd.DataFrame(filas).sort_values("A2").reset_index(drop=True); ranking.index += 1
ranking.round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].hist(x, bins=20, density=True, color=GRIS, alpha=.35, edgecolor="white")
t = np.linspace(0.01, x.max(), 300)
for (nombre, (dist, fijo)), col in zip(list(candidatas.items())[:4], PAL):
    ax[0].plot(t, dist.pdf(t, *dist.fit(x, **fijo)), color=col, lw=2, label=nombre)
ax[0].legend(); ax[0].set_title("Densidades ajustadas")
best = candidatas[ranking.iloc[0]["distribución"]]; par = best[0].fit(x, **best[1])
stats.probplot(x, dist=best[0], sparams=par, plot=ax[1]); ax[1].set_title(f"Gráfico P-P/Q-Q: {ranking.iloc[0]['distribución']}")
plt.show()

### Exportar a la sintaxis de los simuladores
| Software | Exponencial | Gamma | Triangular | Normal |
|---|---|---|---|---|
| ProModel | `E(media)` | `G(α, β)` | `T(a, c, b)` | `N(μ, σ)` |
| FlexSim | `exponential(0, media, stream)` | `gamma(0, β, α, stream)`* | `triangular(a, b, c, stream)` | `normal(μ, σ, stream)` |
| Excel | `=-media*LN(1-RAND())` | `=GAMMA.INV(RAND(), α, β)` | fórmula por tramos | `=NORM.INV(RAND(), μ, σ)` |

\* En FlexSim verifica en la ayuda el orden escala/forma de tu versión.

In [ ]:
a_, _, b_ = stats.gamma.fit(x, floc=0); m_ = x.mean()
print("ProModel : G(%.3f, %.3f)     o   E(%.3f)" % (a_, b_, m_))
print("FlexSim  : gamma(0, %.3f, %.3f, getstream(current))" % (b_, a_))
print("Excel    : =GAMMA.INV(RAND(), %.3f, %.3f)" % (a_, b_))

## 3.2 Líneas de espera: portería vehicular M/M/c

Llegan $\lambda = 50$ vehículos/h; cada carril atiende $\mu = 60$/h. Costos: \$30 000 por hora de espera de un vehículo y \$25 000 por hora de carril abierto.

Fórmula de Erlang C con $a = \lambda/\mu$, $\rho = a/c$:
$$P_w = \frac{\frac{a^c}{c!}\frac{1}{1-\rho}}{\sum_{k=0}^{c-1}\frac{a^k}{k!} + \frac{a^c}{c!}\frac{1}{1-\rho}}, \quad W_q = \frac{P_w}{c\mu - \lambda}, \quad L_q = \lambda W_q$$

In [ ]:
def erlang_c(lam, mu, c):
    a = lam / mu; rho = a / c
    if rho >= 1: return None
    suma = sum(a ** k / math.factorial(k) for k in range(c)); ult = a ** c / math.factorial(c) / (1 - rho)
    Pw = ult / (suma + ult); Wq = Pw / (c * mu - lam)
    return dict(c=c, rho=rho, Pw=Pw, Wq_min=Wq * 60, W_min=(Wq + 1 / mu) * 60, Lq=lam * Wq, L=lam * (Wq + 1 / mu))

lam, mu, cw, cs = 50, 60, 30000, 25000
filas = []
for c in (1, 2, 3, 4):
    e = erlang_c(lam, mu, c)
    sims = [simular_cola(20000, lam, mu, np.random.default_rng(900 + 10 * c + k), c=c) for k in range(3)]
    e["Wq_sim_min"] = np.mean([s["Wq"] for s in sims]) * 60
    e["costo_$/h"] = cs * c + cw * e["Lq"]; filas.append(e)
porteria = pd.DataFrame(filas).set_index("c"); display(porteria.round(4))
print(f"Número óptimo de carriles: c = {porteria['costo_$/h'].idxmin()}")
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(porteria.index, porteria["costo_$/h"], color=[VERDE if c == porteria['costo_$/h'].idxmin() else MORADO for c in porteria.index]); ax[0].set_title("Costo total por hora"); ax[0].set_xlabel("carriles")
ax[1].plot(porteria.index, porteria.Wq_min, "o-", label="Erlang C"); ax[1].plot(porteria.index, porteria.Wq_sim_min, "s--", label="simulación"); ax[1].set_yscale("log"); ax[1].legend(); ax[1].set_title("Wq (min): teoría vs simulación")
plt.show()

In [ ]:
# Evolución de la cola en una corrida (c = 1) para ver el régimen transitorio
s = simular_cola(400, lam, mu, np.random.default_rng(1), c=1)
eventos = np.r_[s["llegada"], s["ini"]]; signo = np.r_[np.ones(400), -np.ones(400)]
o = np.argsort(eventos, kind="stable"); cola = np.cumsum(signo[o])
fig, ax = plt.subplots(); ax.step(eventos[o], cola, where="post", color=MORADO); ax.set_xlabel("horas"); ax.set_ylabel("vehículos en cola"); ax.set_title("Cola con un carril (ρ = 0.83)"); plt.show()

## 3.3 Inventario con política (Q, R)

Cada día: llega el pedido pendiente → se genera la demanda (discreta, por transformada inversa) → se vende lo posible y se registra el faltante → si inventario ≤ R y no hay pedido en camino, se pide Q con tiempo de entrega aleatorio.

Costos: pedido \$100, mantener \$1/unidad/día, faltante \$5/unidad.

In [ ]:
dem_v, dem_p = [6, 8, 10, 12, 14], [0.10, 0.25, 0.35, 0.20, 0.10]
lt_v, lt_p = [1, 2, 3], [0.3, 0.5, 0.2]
print("Demanda: acumulada", np.cumsum(dem_p), "→ media", np.dot(dem_v, dem_p))
filas, costo = simular_inventario(30, 60, 25, 50, dem_v, dem_p, lt_v, lt_p, 100, 1, 5, np.random.default_rng(5), detalle=True)
display(pd.DataFrame(filas).round(4).head(15)); print(costo)

In [ ]:
Qs, Rs = [30, 40, 50, 60, 80, 100], [15, 20, 25, 30, 35]
mapa = pd.DataFrame([[np.mean([simular_inventario(360, Q, Rr, 50, dem_v, dem_p, lt_v, lt_p, 100, 1, 5, np.random.default_rng(7000 + k))["total"] / 360 for k in range(15)])
                      for Q in Qs] for Rr in Rs], index=pd.Index(Rs, name="R"), columns=pd.Index(Qs, name="Q"))
fig, ax = plt.subplots(figsize=(8, 4.2)); sns.heatmap(mapa, annot=True, fmt=".1f", cmap="viridis_r", ax=ax); ax.set_title("Costo promedio diario ($)"); plt.show()
q, r_ = mapa.stack().idxmin()[::-1]
print(f"Mejor política en la malla: Q = {q}, R = {r_}  (costo ≈ ${mapa.loc[r_, q]:.1f}/día)")

**Interpretación:** Q grande ahorra pedidos pero dispara el costo de mantener (columna derecha). La demanda durante el tiempo de entrega efectivo (1 a 3 días más el día de llegada) promedia unas 29 unidades; aun así el R óptimo de la malla es bajo porque el faltante es barato (\$5/unidad) frente a mantener inventario. **Prueba:** sube el costo de faltante a \$20 y observa cómo el mejor R se desplaza hacia arriba.

## 3.4 Línea de producción: 3 estaciones en serie

Llegadas exponenciales cada 5.5 min en promedio; tiempos triangulares (min): corte Tri(3, 4, 6), ensamble Tri(4, 5, 7), pintura Tri(2, 3, 5).

In [ ]:
def linea(n, media_llegada, estaciones, rng):
    llegada = np.cumsum(rng.exponential(media_llegada, n)); t_prev = llegada.copy(); ocupado = []; esperas = []
    for (a, c, b) in estaciones:
        sv = inv_triangular(rng.random(n), a, c, b); fin = np.empty(n); libre = 0.0; esp = np.empty(n)
        for i in range(n):
            ini = max(t_prev[i], libre); esp[i] = ini - t_prev[i]; fin[i] = ini + sv[i]; libre = fin[i]
        ocupado.append(sv.sum()); esperas.append(esp.mean()); t_prev = fin
    T = t_prev[-1] - llegada[0]; ct = t_prev - llegada
    return dict(util=[o / T for o in ocupado], espera=esperas, ciclo=ct.mean(), throughput_h=n / T * 60, wip=ct.sum() / T)

est = [(3, 4, 6), (4, 5, 7), (2, 3, 5)]
res = [linea(5000, 5.5, est, np.random.default_rng(60 + k)) for k in range(10)]
t = pd.DataFrame({"estación": ["Corte", "Ensamble", "Pintura"], "media servicio": [sum(e) / 3 for e in est],
                  "ρ teórica = media/5.5": [sum(e) / 3 / 5.5 for e in est],
                  "utilización simulada": np.mean([r["util"] for r in res], 0), "espera media (min)": np.mean([r["espera"] for r in res], 0)})
display(t.round(3))
print(f"Tiempo de ciclo = {np.mean([r['ciclo'] for r in res]):.1f} min;  throughput = {np.mean([r['throughput_h'] for r in res]):.2f} piezas/h;  WIP = {np.mean([r['wip'] for r in res]):.1f}")
print("Cuello de botella: ENSAMBLE (ρ ≈ 0.97) → la cola se acumula antes de esa estación.")

## 3.5 Tiempo de entrega de un pedido (Monte Carlo)

Etapas: alistamiento Tri(1, 2, 4) d, transporte Tri(2, 3, 6) d, aduana Tri(0.5, 1, 3) d. ¿Probabilidad de entregar en menos de 7 días?

In [ ]:
etapas = [(1, 2, 4), (2, 3, 6), (0.5, 1, 3)]
tt = tiempo_pedido(np.random.default_rng(21).random((10000, 3)), etapas)
p7 = (tt <= 7).mean(); e = 1.96 * math.sqrt(p7 * (1 - p7) / len(tt))
print(f"media = {tt.mean():.3f} d (teórica {sum(sum(e_) / 3 for e_ in etapas):.3f});  P(T ≤ 7) = {p7:.4f} ± {e:.4f};  percentil 90 = {np.percentile(tt, 90):.2f} d")
fig, ax = plt.subplots(); sns.histplot(tt, bins=50, stat="density", color=AZUL, ax=ax); ax.axvline(7, color=CORAL, ls="--"); ax.set_title("Distribución del tiempo total de entrega"); plt.show()

---
# Parte 4 · Casos de aplicación

Para cada caso: **(1)** qué variable aleatoria aparece, **(2)** qué distribución y por qué, **(3)** sus parámetros (media, varianza, asimetría, curtosis), **(4)** una simulación y **(5)** una pregunta de decisión.

In [ ]:
def ficha(nombre, dist, descripcion):
    m, v, s, k = dist.stats(moments="mvsk")
    return dict(caso=nombre, variable=descripcion, media=float(m), varianza=float(v), desv=float(np.sqrt(v)), asimetría=float(s), curtosis_exceso=float(k))
fichas = []

## 4.1 Agrotrónica: invernadero y riego automático

| Variable | Distribución | Parámetros |
|---|---|---|
| Humedad del suelo (%) | Normal | μ = 32, σ = 4 |
| Lluvias por semana | Poisson | λ = 2.4 |
| Vida de la bomba (h) | Weibull | k = 1.8, λ = 900 |

El riego se activa cuando la humedad < 25 %.

In [ ]:
hum, lluvia, bomba = stats.norm(32, 4), stats.poisson(2.4), stats.weibull_min(1.8, scale=900)
fichas += [ficha("Agrotrónica", hum, "humedad %"), ficha("Agrotrónica", lluvia, "lluvias/semana"), ficha("Agrotrónica", bomba, "vida bomba h")]
print(f"P(humedad < 25) = Φ((25−32)/4) = Φ(−1.75) = {hum.cdf(25):.4f}  → ≈ {hum.cdf(25) * 365:.0f} días de riego al año")
print(f"P(semana sin lluvia) = e^(−2.4) = {lluvia.pmf(0):.4f};  P(≥ 3 lluvias) = {lluvia.sf(2):.4f}")
print(f"P(bomba falla antes de 1000 h) = {bomba.cdf(1000):.4f};  MTTF = {bomba.mean():.1f} h")

# Temporada de 120 días: humedad con memoria (AR(1)) alrededor de 29 % por evaporación; cada lluvia suma 3 %, y el riego suma 6 %
rng = np.random.default_rng(42); dias = 120; h = np.empty(dias); h[0] = 32; riego = np.zeros(dias, bool)
for d in range(1, dias):
    lluvias_hoy = rng.poisson(2.4 / 7)
    h[d] = 29 + 0.6 * (h[d - 1] - 29) + rng.normal(0, 4 * math.sqrt(1 - .36)) + 3 * lluvias_hoy
    if h[d] < 25: riego[d] = True; h[d] += 6
fig, ax = plt.subplots(); ax.plot(h, color=VERDE); ax.scatter(np.where(riego)[0], h[riego], color=CORAL, zorder=3, label="riego activado")
ax.axhline(25, color=CORAL, ls="--"); ax.set_xlabel("día"); ax.set_ylabel("humedad (%)"); ax.legend(); ax.set_title("Temporada simulada del invernadero"); plt.show()
print(f"Riegos en la temporada: {riego.sum()}")

## 4.2 Domótica: hogar inteligente

* **Dispositivos inteligentes por hogar** → **serie logarítmica** (θ = 0.8): $p(k) = \dfrac{-\theta^k}{k\ln(1-\theta)}$, $k = 1, 2, \dots$; $E[X] = \dfrac{-\theta}{(1-\theta)\ln(1-\theta)}$.
  Modela conteos donde casi todos tienen pocos y unos pocos tienen muchos.
* **Eventos de automatización por hora (noche)** → Poisson (λ = 6).
* **Consumo diario (kWh)** → lognormal (μ = 2.3, σ = 0.35): $E = e^{\mu + \sigma^2/2}$.

In [ ]:
th = 0.8; k = np.arange(1, 13)
p_manual = -th ** k / (k * math.log(1 - th))
t = pd.DataFrame({"k": k, "p(k) manual": p_manual, "p(k) SciPy": stats.logser.pmf(k, th), "F(k)": np.cumsum(p_manual)})
display(t.round(5))
media = -th / ((1 - th) * math.log(1 - th)); var = -th * (th + math.log(1 - th)) / ((1 - th) ** 2 * math.log(1 - th) ** 2)
print(f"E[X] = {media:.4f} (SciPy {stats.logser.mean(th):.4f});  Var = {var:.4f} (SciPy {stats.logser.var(th):.4f})")

# Generación por transformada inversa (búsqueda secuencial con recurrencia p(k+1) = p(k)·θ·k/(k+1))
def logser_inv(r, th):
    k, p = 1, -th / math.log(1 - th); F = p
    while r > F: p *= th * k / (k + 1); k += 1; F += p
    return k
muestra = np.array([logser_inv(r, th) for r in np.random.default_rng(12).random(5000)])
print(f"Media simulada = {muestra.mean():.3f}")
res = chi2_discreta(muestra, lambda x: stats.logser.pmf(x, th), list(range(1, 10)))
decision(res["chi0"], res["crit"], "χ²₀ serie logarítmica")
fichas.append(ficha("Domótica", stats.logser(th), "dispositivos/hogar"))

In [ ]:
cons = stats.lognorm(0.35, scale=math.exp(2.3)); ev = stats.poisson(6)
fichas += [ficha("Domótica", cons, "consumo kWh/día"), ficha("Domótica", ev, "eventos/hora")]
print(f"Consumo: media = e^(2.3 + 0.35²/2) = {cons.mean():.2f} kWh;  P(> 15 kWh) = {cons.sf(15):.4f};  percentil 95 = {cons.ppf(.95):.2f}")
print(f"Eventos: P(más de 10 en una hora) = {ev.sf(10):.4f}")
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(k, p_manual, color=AZUL); ax[0].set_title("Serie logarítmica (θ = 0.8)"); ax[0].set_xlabel("dispositivos")
t = np.linspace(0, 30, 300); ax[1].plot(t, cons.pdf(t), color=MORADO, lw=2); ax[1].fill_between(t, cons.pdf(t), where=t > 15, color=CORAL, alpha=.4); ax[1].set_title("Consumo diario lognormal"); ax[1].set_xlabel("kWh")
plt.show()

## 4.3 Inmótica: edificio inteligente

* Ascensores en hora pico: llegadas Poisson λ = 180 personas/h; cada ascensor transporta μ = 75 personas/h → M/M/c.
* Ocupación de 12 salas, cada una ocupada con p = 0.65 → Binomial(12, 0.65).
* Temperatura interior: Normal(22, 0.8); el HVAC actúa fuera de ±1.5 °C.

In [ ]:
filas = []
for c in (3, 4, 5):
    e = erlang_c(180, 75, c); filas.append(dict(ascensores=c, rho=e["rho"], P_esperar=e["Pw"], Wq_s=e["Wq_min"] * 60))
display(pd.DataFrame(filas).round(4))
salas, temp = stats.binom(12, .65), stats.norm(22, .8)
print(f"P(todas las salas ocupadas) = 0.65^12 = {salas.pmf(12):.4f};  E[salas ocupadas] = {salas.mean():.1f}")
print(f"P(HVAC actúa) = P(|T − 22| > 1.5) = {2 * temp.sf(23.5):.4f}")
fichas += [ficha("Inmótica", salas, "salas ocupadas"), ficha("Inmótica", temp, "temperatura °C")]

## 4.4 Visión computacional: lectura de placas con YOLO (simulación)

Un sistema real tiene dos etapas: **(1)** un detector YOLO encuentra la placa en el cuadro y **(2)** un OCR lee los caracteres. Cada etapa genera variables aleatorias:

| Variable | Distribución | Por qué |
|---|---|---|
| Vehículos por minuto | Poisson (λ = 3) | llegadas independientes |
| Tiempo entre vehículos | Exponencial (media 20 s) | consecuencia de la Poisson |
| Confianza de la detección | Beta (18, 2.2) | valor en [0, 1] concentrado cerca de 1 |
| Intentos de lectura hasta el éxito | Geométrica (p = 0.85) | ensayos independientes |
| Latencia de inferencia (ms) | Lognormal (ln 28, 0.25) | positiva y asimétrica |
| Error de posición del recuadro (px) | Normal (0, 3) | suma de muchos errores pequeños |
| Placas repetidas por vehículo en un día | Serie logarítmica (θ = 0.6) | muchos pasan 1 vez, pocos muchas |

> **Privacidad:** una placa es un dato personal (Ley 1581 de 2012 en Colombia). Aquí todas las placas son **sintéticas**. Con video real se requiere autorización, aviso de captura y política de tratamiento.

In [ ]:
rng = np.random.default_rng(77)
letras = np.array(list("ABCDEFGHJKLMNPRSTUVWXYZ"))
def placa_sintetica(rng): return "".join(rng.choice(letras, 3)) + "".join(map(str, rng.integers(0, 10, 3)))

minutos = 240
llegadas_min = rng.poisson(3, minutos); n = llegadas_min.sum()
yolo = pd.DataFrame({
    "minuto": np.repeat(np.arange(minutos), llegadas_min),
    "placa": [placa_sintetica(rng) for _ in range(n)],
    "confianza": rng.beta(18, 2.2, n),
    "intentos_ocr": rng.geometric(0.85, n),
    "latencia_ms": rng.lognormal(math.log(28), 0.25, n),
    "error_px": rng.normal(0, 3, n),
})
yolo["aceptada"] = yolo.confianza >= 0.80
display(yolo.head(10).round(3))
print(f"{n} vehículos en {minutos} min;  detecciones aceptadas (conf ≥ 0.80): {yolo.aceptada.mean():.1%};  FPS posibles = {1000 / yolo.latencia_ms.mean():.1f}")

In [ ]:
# Análisis de distribuciones: parámetros estimados vs. teóricos y prueba de bondad para cada variable
filas = []
# Poisson: conteos por minuto
lam_ = llegadas_min.mean(); r_ = chi2_discreta(llegadas_min, lambda x: stats.poisson.pmf(x, lam_), list(range(0, 9)), p_estimados=1)
filas.append(dict(variable="vehículos/min", distribución="Poisson", estimado=f"λ = {lam_:.3f}", teórico="λ = 3", prueba="χ²", estadístico=r_["chi0"], p=r_["p"]))
# Exponencial: tiempo entre llegadas dentro de la hora (aproximado con llegadas uniformes en el minuto)
tiempos = np.sort(yolo.minuto.values * 60 + rng.random(n) * 60); ia = np.diff(tiempos)
filas.append(dict(variable="tiempo entre vehículos (s)", distribución="Exponencial", estimado=f"media = {ia.mean():.2f}", teórico="media = 20", prueba="K-S", estadístico=stats.kstest(ia, "expon", args=(0, ia.mean())).statistic, p=stats.kstest(ia, "expon", args=(0, ia.mean())).pvalue))
# Beta: confianza
a_, b_, _, _ = stats.beta.fit(yolo.confianza, floc=0, fscale=1); ad = anderson_darling(yolo.confianza.values, lambda v: stats.beta.cdf(v, a_, b_))
filas.append(dict(variable="confianza", distribución="Beta", estimado=f"α = {a_:.2f}, β = {b_:.2f}", teórico="α = 18, β = 2.2", prueba="A-D", estadístico=ad["A2"], p=np.nan))
# Geométrica: intentos
p_ = 1 / yolo.intentos_ocr.mean(); r_ = chi2_discreta(yolo.intentos_ocr.values, lambda x: stats.geom.pmf(x, p_), [1, 2, 3], p_estimados=1)
filas.append(dict(variable="intentos OCR", distribución="Geométrica", estimado=f"p = {p_:.3f}", teórico="p = 0.85", prueba="χ²", estadístico=r_["chi0"], p=r_["p"]))
# Lognormal: latencia
s_, _, sc_ = stats.lognorm.fit(yolo.latencia_ms, floc=0); kk = stats.kstest(yolo.latencia_ms, "lognorm", args=(s_, 0, sc_))
filas.append(dict(variable="latencia (ms)", distribución="Lognormal", estimado=f"μ = {math.log(sc_):.3f}, σ = {s_:.3f}", teórico=f"μ = {math.log(28):.3f}, σ = 0.25", prueba="K-S", estadístico=kk.statistic, p=kk.pvalue))
# Normal: error de posición
mu_, sd_ = yolo.error_px.mean(), yolo.error_px.std(ddof=1); A = anderson_darling(yolo.error_px.values, lambda v: stats.norm.cdf(v, mu_, sd_))["A2"] * (1 + .75 / n + 2.25 / n ** 2)
filas.append(dict(variable="error de posición (px)", distribución="Normal", estimado=f"μ = {mu_:.3f}, σ = {sd_:.3f}", teórico="μ = 0, σ = 3", prueba="A* (crít. 0.752)", estadístico=A, p=np.nan))
display(pd.DataFrame(filas).round(4))
print("Nota: el p de K-S con parámetros estimados es optimista (ver Lilliefors, 1.7).")

In [ ]:
# Serie logarítmica: cuántas veces pasa cada placa en un día (flota de visitantes y residentes)
visitas = np.array([logser_inv(r, 0.6) for r in np.random.default_rng(78).random(800)])
print(f"Placas distintas = 800; pasos totales = {visitas.sum()};  media = {visitas.mean():.3f} (teórica {stats.logser.mean(.6):.3f});  P(1 sola vez) = {(visitas == 1).mean():.3f} (teórica {stats.logser.pmf(1, .6):.3f})")
fichas += [ficha("YOLO", stats.poisson(3), "vehículos/min"), ficha("YOLO", stats.beta(18, 2.2), "confianza"), ficha("YOLO", stats.geom(.85), "intentos OCR"),
           ficha("YOLO", stats.lognorm(.25, scale=28), "latencia ms"), ficha("YOLO", stats.logser(.6), "pasos por placa/día")]
fig, ax = plt.subplots(2, 3, figsize=(15, 7.5)); ax = ax.ravel()
sns.countplot(x=llegadas_min, ax=ax[0], color=MORADO); ax[0].set_title("Vehículos por minuto (Poisson)")
ax[1].hist(yolo.confianza, bins=30, density=True, color=VERDE, alpha=.6); t = np.linspace(.5, 1, 200); ax[1].plot(t, stats.beta.pdf(t, 18, 2.2), color=TINTA); ax[1].axvline(.8, color=CORAL, ls="--"); ax[1].set_title("Confianza (Beta)")
sns.countplot(x=yolo.intentos_ocr, ax=ax[2], color=AMBAR); ax[2].set_title("Intentos de OCR (geométrica)")
ax[3].hist(yolo.latencia_ms, bins=40, density=True, color=AZUL, alpha=.6); t = np.linspace(10, 60, 200); ax[3].plot(t, stats.lognorm.pdf(t, .25, scale=28), color=TINTA); ax[3].set_title("Latencia (lognormal)")
ax[4].hist(yolo.error_px, bins=40, density=True, color=CORAL, alpha=.6); t = np.linspace(-12, 12, 200); ax[4].plot(t, stats.norm.pdf(t, 0, 3), color=TINTA); ax[4].set_title("Error de posición (normal)")
sns.countplot(x=np.minimum(visitas, 8), ax=ax[5], color=GRIS); ax[5].set_title("Pasos por placa (serie log., 8 = 8+)")
plt.tight_layout(); plt.show()

In [ ]:
# Pregunta de decisión: ¿qué umbral de confianza usar?
umbrales = np.arange(.70, .96, .025)
pd.DataFrame({"umbral": umbrales, "aceptadas": [(yolo.confianza >= u).mean() for u in umbrales],
              "teórico Beta": [stats.beta.sf(u, 18, 2.2) for u in umbrales]}).round(4)

### 4.5 (Opcional) YOLO real con ultralytics

Esta celda **no se ejecuta** por defecto. Para usarla en Colab:
1. Cambia `EJECUTAR_YOLO_REAL = True`.
2. Sube un video propio **con autorización** (por ejemplo `porteria.mp4`).
3. El modelo general `yolov8n.pt` detecta vehículos (clases COCO 2, 3, 5, 7). Para placas necesitas un modelo entrenado para placas (por ejemplo, uno propio entrenado con un conjunto de placas sintéticas o con autorización).

El resultado alimenta exactamente el análisis anterior: conteos por minuto, confianzas y latencias reales.

In [ ]:
EJECUTAR_YOLO_REAL = False
if EJECUTAR_YOLO_REAL:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
    from ultralytics import YOLO
    modelo = YOLO("yolov8n.pt")
    registros = []
    for i, r in enumerate(modelo.predict(source="porteria.mp4", stream=True, classes=[2, 3, 5, 7], conf=0.25, verbose=False)):
        for b in r.boxes:
            registros.append(dict(cuadro=i, clase=int(b.cls), confianza=float(b.conf), latencia_ms=r.speed["inference"]))
    reales = pd.DataFrame(registros)
    display(reales.describe())
    a_, b_, _, _ = stats.beta.fit(reales.confianza.clip(1e-4, 1 - 1e-4), floc=0, fscale=1)
    print(f"Confianza real ~ Beta({a_:.2f}, {b_:.2f})")
else:
    print("Sección opcional desactivada (EJECUTAR_YOLO_REAL = False).")

## 4.6 Encuesta semiestructurada: energía renovable

Una encuesta **semiestructurada** combina preguntas cerradas (Likert, sí/no, opción múltiple) con preguntas abiertas que luego se **codifican** en categorías. Simulamos 300 respuestas con distribuciones plausibles:

| Pregunta | Tipo | Modelo de simulación |
|---|---|---|
| Estrato | categórica | multinomial |
| «¿Instalaría paneles solares?» | Likert 1–5 | multinomial que depende del estrato |
| Gasto mensual en energía | numérica | lognormal |
| Disposición a pagar por kit solar | numérica | normal truncada |
| «¿Qué le impide instalar?» (abierta) | texto → códigos | multinomial sobre categorías |

In [ ]:
rng = np.random.default_rng(2027); N = 300
estrato = rng.choice(["1-2", "3-4", "5-6"], N, p=[.35, .40, .25])
prob_likert = {"1-2": [.06, .10, .22, .37, .25], "3-4": [.04, .08, .18, .38, .32], "5-6": [.03, .07, .17, .35, .38]}
likert = np.array([rng.choice([1, 2, 3, 4, 5], p=prob_likert[e]) for e in estrato])
gasto = rng.lognormal(np.log(np.select([estrato == "1-2", estrato == "3-4"], [90, 150], 260) * 1000), .35)
pagar = np.clip(rng.normal(2.5, 1.0, N) + (likert - 3) * .4, 0, None)  # millones COP
barreras = ["costo inicial", "no sabe cómo", "vive en arriendo", "desconfía de la tecnología", "ninguna"]
texto = {"costo inicial": "es muy caro instalarlo", "no sabe cómo": "no sé a quién acudir", "vive en arriendo": "la casa no es mía",
         "desconfía de la tecnología": "no creo que funcione bien", "ninguna": "lo haría ya"}
barrera = rng.choice(barreras, N, p=[.42, .18, .17, .10, .13])
enc = pd.DataFrame({"estrato": estrato, "instalaria_1a5": likert, "gasto_mensual_cop": gasto.round(-2), "pagaria_millones": pagar.round(2),
                    "respuesta_abierta": [texto[b] for b in barrera], "codigo_barrera": barrera})
display(enc.head(8))

In [ ]:
# 1) Distribución de la escala Likert y proporción favorable (4 o 5) con IC 95 %
conteo = enc.instalaria_1a5.value_counts().sort_index(); p_fav = (enc.instalaria_1a5 >= 4).mean(); e = 1.96 * math.sqrt(p_fav * (1 - p_fav) / N)
print(conteo.to_string()); print(f"Media Likert = {enc.instalaria_1a5.mean():.2f};  favorable = {p_fav:.3f}  IC 95 % = [{p_fav - e:.3f}, {p_fav + e:.3f}]")
# 2) Independencia estrato × disposición (agrupando 1-2, 3, 4-5)
grupo = pd.cut(enc.instalaria_1a5, [0, 2, 3, 5], labels=["No", "Indeciso", "Sí"])
ct = pd.crosstab(enc.estrato, grupo); chi, p, gl, _ = stats.chi2_contingency(ct)
display(ct); print(f"χ² = {chi:.3f}, gl = {gl}, p = {p:.4f}")
# 3) Gasto: ¿lognormal?  4) Barreras codificadas
s_, _, sc_ = stats.lognorm.fit(enc.gasto_mensual_cop, floc=0)
print(f"Gasto ~ Lognormal(μ = {math.log(sc_):.3f}, σ = {s_:.3f}) (mezcla de estratos → σ mayor que 0.35)")
display(enc.codigo_barrera.value_counts(normalize=True).round(3).to_frame("proporción"))
# 5) Tamaño de muestra para error 5 % y 3 %
for err in (.05, .03): print(f"n requerido (p = 0.5, 95 %, e = {err:.0%}) = {math.ceil(1.96 ** 2 * .25 / err ** 2)}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
pd.crosstab(enc.estrato, enc.instalaria_1a5, normalize="index").plot(kind="barh", stacked=True, ax=ax[0], color=["#E4572E", "#F2A900", "#C9C3DD", "#3A86C8", "#00A896"], legend=False); ax[0].set_title("Disposición a instalar por estrato")
sns.boxplot(data=enc, x="estrato", y="pagaria_millones", order=["1-2", "3-4", "5-6"], ax=ax[1], color=VERDE); ax[1].set_title("Disposición a pagar (millones COP)")
enc.codigo_barrera.value_counts().plot(kind="bar", ax=ax[2], color=MORADO); ax[2].set_title("Barreras (respuesta abierta codificada)"); ax[2].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

## 4.7 Ficha resumen de todas las distribuciones usadas

In [ ]:
pd.DataFrame(fichas).round(4)

---
# Parte 5 · Los 18 ejercicios de la presentación, resueltos

Cada celda calcula paso a paso la solución que aparece en las notas del orador.

In [ ]:
print("EJERCICIO 1 · χ² discreta")
O = np.array([10, 22, 18, 10]); p = np.array([.2, .3, .3, .2]); E = 60 * p
display(pd.DataFrame({"O": O, "p": p, "E": E, "(O−E)²/E": (O - E) ** 2 / E}))
decision(((O - E) ** 2 / E).sum(), stats.chi2.ppf(.95, 3), "χ²₀ (3 gl)")

print("\nEJERCICIO 2 · límites equiprobables de Exp(5) con 5 clases")
print([round(-5 * math.log(1 - j / 5), 3) for j in range(1, 5)])

print("\nEJERCICIO 3 · independencia 2×2")
t = np.array([[40, 20], [30, 30]]); E = np.outer(t.sum(1), t.sum(0)) / t.sum()
print("E =", E.tolist()); decision(((t - E) ** 2 / E).sum(), 3.84, "χ²₀")

print("\nEJERCICIO 4 · K-S frente a Exp(3)")
r = ks([0.8, 1.9, 2.6, 4.4, 7.1], lambda x: 1 - np.exp(-np.asarray(x) / 3))
display(pd.DataFrame({"x": r["x"], "F0": r["F"], "D+": r["dmas"], "D−": r["dmenos"]}).round(4)); decision(r["D"], 0.565, "D")

print("\nEJERCICIO 5 · A-D normal estimada, n = 40")
A = 0.70 * (1 + .75 / 40 + 2.25 / 1600); decision(A, 0.752, "A*")

print("\nEJERCICIO 6 · potencia con n = 15")
print(tasa_rechazo(lambda r, n: r.gamma(2.5, 1.8, n), 15, reps=600, semilla=6), "← (χ², K-S, A-D): rechazos bajos = potencia baja")

In [ ]:
print("EJERCICIO 7 · Weibull k = 2, λ = 500:", [round(500 * (-math.log(1 - r)) ** .5, 1) for r in (.30, .90)])
print("EJERCICIO 8 · geométrica p = 0.7:", [(round(math.log(1 - r) / math.log(.3), 3), math.ceil(math.log(1 - r) / math.log(.3))) for r in (.5, .95)])
print("EJERCICIO 9 · empírica:", empirica_inv(.6, np.array([2, 4, 5, 9, 12])))
print("EJERCICIO 10 · Erlang k = 3, media 9:", round(-3 * math.log(.5 * .4 * .8), 3))
y = -math.log(1 - .70); print(f"EJERCICIO 11 · A-R normal: Y = {y:.3f}; umbral = {math.exp(-(y - 1) ** 2 / 2):.3f} ≥ 0.50 → se acepta")
print(f"EJERCICIO 12 · composición: r1 = 0.85 ≥ 0.8 → componente 2; x = {-10 * math.log(1 - .6):.3f}; media = {.8 * 1 + .2 * 10}")

In [ ]:
print("EJERCICIO 13 · portería λ = 50, μ = 30 por carril")
for c in (1, 2, 3):
    e = erlang_c(50, 30, c); print(f"  c = {c}: " + ("inestable (ρ ≥ 1)" if e is None else f"ρ = {e['rho']:.3f}, Wq = {e['Wq_min']:.2f} min"))
print("EJERCICIO 14 · ver el mapa (Q, R): con Q grande sube el inventario promedio y el costo de mantener domina:")
print(mapa.loc[25].round(1).to_string())
print("EJERCICIO 15 · nuevas medias:", {n_: round(sum(e) / 3, 2) for n_, e in zip(["corte", "ensamble", "pintura"], [(3, 4, 6), (3, 4, 6), (2, 3, 5)])}, "→ corte y ensamble empatan (ρ ≈ 0.79)")
r15 = np.mean([linea(5000, 5.5, [(3, 4, 6), (3, 4, 6), (2, 3, 5)], np.random.default_rng(80 + k))["util"] for k in range(5)], 0)
print("   utilizaciones simuladas:", np.round(r15, 3))
print(f"EJERCICIO 16 · umbral del 2 %: 32 + z(0.02)·4 = 32 + ({stats.norm.ppf(.02):.3f})·4 = {stats.norm.ppf(.02, 32, 4):.2f} %")
print(f"EJERCICIO 17 · P(leer en ≤ 2 intentos) = 1 − 0.15² = {stats.geom.cdf(2, .85):.4f}")
print(f"EJERCICIO 18 · n = 1.96²·0.25/0.03² = {1.96 ** 2 * .25 / .03 ** 2:.1f} → {math.ceil(1.96 ** 2 * .25 / .03 ** 2)}")

---
## Para seguir practicando

1. Cambia la semilla de la sección 1.2 y observa cuántas veces (de 20) se rechaza la Poisson: debería ser ≈ 1.
2. En 3.1 agrega la distribución `stats.erlang` y la triangular al ranking.
3. En 3.2 cambia el costo de espera a \$60 000/h: ¿cambia el número óptimo de carriles?
4. En 4.4 sube la latencia a lognormal(ln 45, 0.3): ¿cuántos FPS se pierden?
5. En 4.6 reemplaza los datos simulados por tu propia encuesta (un CSV con las mismas columnas) y repite el análisis.